# Lightweight Budget Forecasting

**Objective:** Predict next month's spending from WealthWise transaction history.

**Design:**
- Uses transaction data already available in WealthWise.
- No external dataset is required.
- Uses a lightweight Random Forest Regressor.
- Uses only simple monthly spending features.
- Falls back to the recent average when there is not enough history.

**Expected columns:** `Date`, `Amount`, `Category`


In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)


## 1. Load Transaction Data

For the real WealthWise application, replace the CSV path with data returned by the backend API/database.

No Kaggle dataset is downloaded by this notebook.


In [ ]:
# ============================================================
# LOAD DATA
# ============================================================
DATA_PATH = "transactions.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Export your WealthWise transactions "
        "to this CSV or replace DATA_PATH with your backend export."
    )

df = pd.read_csv(DATA_PATH)

required = {"Date", "Amount", "Category"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
df["Category"] = df["Category"].astype(str)

df = df.dropna(subset=["Date", "Amount", "Category"]).copy()
df = df.sort_values("Date")

print(f"Transactions: {len(df)}")
print(f"Date range: {df['Date'].min().date()} -> {df['Date'].max().date()}")
print(f"Categories: {sorted(df['Category'].unique())}")


## 2. Monthly Spending

Convert transactions into one row per month and category.


In [ ]:
# ============================================================
# MONTHLY AGGREGATION
# ============================================================
df["date"] = df["Date"].dt.to_period("M").dt.to_timestamp()

monthly = (
    df.groupby(["date", "Category"], as_index=False)
      .agg(
          total_amount=("Amount", "sum"),
          transaction_count=("Amount", "count")
      )
      .sort_values(["Category", "date"])
      .reset_index(drop=True)
)

print(monthly.head())
print(f"Monthly rows: {len(monthly)}")


## 3. Simple Features

For each category, use the previous 1, 2 and 3 months plus the recent average.

The target is the spending amount for the current month. This creates a simple supervised forecasting problem.


In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================
def create_features(group):
    group = group.sort_values("date").copy()

    # Previous spending
    group["lag_1"] = group["total_amount"].shift(1)
    group["lag_2"] = group["total_amount"].shift(2)
    group["lag_3"] = group["total_amount"].shift(3)

    # Recent average, excluding current month
    group["avg_3"] = group["total_amount"].shift(1).rolling(3).mean()

    # Previous transaction count
    group["count_1"] = group["transaction_count"].shift(1)

    return group

features = (
    monthly.groupby("Category", group_keys=False)
           .apply(create_features)
           .reset_index(drop=True)
)

features = features.dropna(
    subset=["lag_1", "lag_2", "lag_3", "avg_3", "count_1"]
).reset_index(drop=True)

feature_cols = ["lag_1", "lag_2", "lag_3", "avg_3", "count_1"]
target_col = "total_amount"

print(f"Training rows: {len(features)}")
print(features[["date", "Category"] + feature_cols + [target_col]].head())


## 4. Time-Based Train/Test Split

The model is trained only on older months and tested on newer months. This avoids randomly mixing future data into training.


In [ ]:
# ============================================================
# TIME SPLIT
# ============================================================
unique_dates = sorted(features["date"].unique())

if len(unique_dates) < 5:
    raise ValueError(
        "Not enough monthly history for ML forecasting. "
        "At least 5 months of transaction history is recommended."
    )

split_index = max(1, int(len(unique_dates) * 0.8))
split_date = unique_dates[split_index - 1]

train = features[features["date"] <= split_date]
test = features[features["date"] > split_date]

if test.empty:
    # Keep the final month as test data
    last_date = unique_dates[-1]
    test = features[features["date"] == last_date]
    train = features[features["date"] < last_date]

X_train = train[feature_cols]
y_train = train[target_col]

X_test = test[feature_cols]
y_test = test[target_col]

print(f"Train rows: {len(train)}")
print(f"Test rows: {len(test)}")
print(f"Train through: {train['date'].max().date()}")
print(f"Test from: {test['date'].min().date()}")


## 5. Train Lightweight Model

Random Forest is sufficient here. There is no need for deep learning, embeddings or a dedicated time-series library.


In [ ]:
# ============================================================
# RANDOM FOREST
# ============================================================
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions) if len(y_test) > 1 else np.nan

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")


## 6. Compare With Simple Average Baseline

The ML model should beat a simple recent-average forecast. If it does not, WealthWise should use the average instead of unnecessarily relying on ML.


In [ ]:
# ============================================================
# BASELINE COMPARISON
# ============================================================
baseline_predictions = test["avg_3"].values

baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Random Forest MAE: {mae:.2f}")
print(f"3-month Average MAE: {baseline_mae:.2f}")

if baseline_mae <= mae:
    print("Baseline is better. Use the 3-month average for this dataset.")
else:
    print("Random Forest performs better than the baseline.")


## 7. Forecast Next Month

Generate one prediction for each category using the latest available transaction history.


In [ ]:
# ============================================================
# NEXT MONTH FORECAST
# ============================================================
latest_rows = []

for category, group in monthly.groupby("Category"):
    group = group.sort_values("date")

    if len(group) < 3:
        # Not enough history: use whatever history is available
        forecast = group["total_amount"].mean()
        method = "average"
    else:
        last_3 = group["total_amount"].tail(3).values

        row = pd.DataFrame([{
            "lag_1": last_3[-1],
            "lag_2": last_3[-2],
            "lag_3": last_3[-3],
            "avg_3": np.mean(last_3),
            "count_1": group["transaction_count"].iloc[-1]
        }])

        if len(group) >= 4:
            forecast = float(model.predict(row[feature_cols])[0])
            method = "random_forest"
        else:
            forecast = float(row["avg_3"].iloc[0])
            method = "average"

    latest_rows.append({
        "Category": category,
        "predicted_next_month": max(0, round(float(forecast), 2)),
        "method": method
    })

forecast_df = pd.DataFrame(latest_rows)

print(forecast_df)


## 8. Actual vs Predicted



In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================
plt.figure(figsize=(12, 5))
plt.plot(y_test.values, label="Actual", marker="o")
plt.plot(predictions, label="Predicted", marker="x")
plt.title("Actual vs Predicted Monthly Spending")
plt.xlabel("Test Month / Category")
plt.ylabel("Spending")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Save Model

Only the model, feature list and basic metadata are required by the backend.


In [ ]:
# ============================================================
# SAVE ARTIFACTS
# ============================================================
joblib.dump(model, os.path.join(ARTIFACTS_DIR, "budget_forecaster.pkl"))

metadata = {
    "model_type": "RandomForestRegressor",
    "features": feature_cols,
    "target": target_col,
    "fallback": "3-month average",
    "categories": sorted(monthly["Category"].unique().tolist())
}

with open(os.path.join(ARTIFACTS_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

forecast_df.to_csv(
    os.path.join(ARTIFACTS_DIR, "next_month_forecast.csv"),
    index=False
)

print(f"Saved model and metadata to: {ARTIFACTS_DIR}")
